# Notebook 4: Exploring Protein Quaternary Structure

## Bio 4525: Structural Bioinformatics of Proteins
**Washington University in St. Louis**

---

## From Tertiary to Quaternary Structure

In Notebooks 1–3 we studied proteins as single polypeptide chains. Many proteins, however, function as **assemblies of two or more chains**. **Quaternary structure** is the arrangement of those chains — how they pack together, what surfaces they share, and how their interactions create functional properties that no single chain could achieve alone.

## Hemoglobin: the Classic Example

Hemoglobin is a **tetramer** of four subunits: two α (alpha) chains and two β (beta) chains, written α₂β₂. Each subunit carries a **heme group** — an iron-containing ring that binds one molecule of oxygen.

### The Molecular Forces That Hold Quaternary Assemblies Together

The same five interactions that fold a single chain into tertiary structure also hold protein-protein interfaces together — but now they act **between** polypeptide chains:

| Interaction | Role at quaternary interfaces |
|---|---|
| **Hydrophobic effect** | Usually the dominant force. Nonpolar residues from two chains cluster at the interface, burying surface away from water — the same driving force that folds each chain individually. |
| **Hydrogen bonds** | Between polar side chains or backbone atoms on opposing chains. Contribute specificity — the two surfaces must match in both shape and polarity to form a stable H-bond network. |
| **Ionic interactions** | Oppositely charged residues (Arg/Lys ↔ Asp/Glu) on different chains attract each other. Especially important at allosteric and regulatory interfaces that must be able to open and close. |
| **Disulfide bonds** | Covalent S–S bonds between Cys residues on **different** chains (e.g., two of insulin's three disulfide bonds cross from its A chain to its B chain). Far stronger than non-covalent contacts; found mainly in secreted proteins where the oxidizing environment favors S–S formation. |
| **Van der Waals** | Weak individually, but hundreds of closely packed atom-atom contacts across a large interface accumulate into a substantial contribution. |

For a protein-protein interaction to be stable, the interface must also have:
- **Buried surface area ≥ ~600–700 Å² per chain** — smaller patches are usually transient or non-specific
- **Chemical complementarity** — polar regions paired with polar, nonpolar with nonpolar
- **Shape complementarity** — the two surfaces fit together geometrically

### Cooperativity and the Hill Equation — A Closer Look

If the four hemoglobin subunits behaved independently, hemoglobin would bind oxygen like four separate myoglobins and would not efficiently release oxygen in the tissues. Instead, hemoglobin is **cooperative** — binding the first oxygen molecule makes the remaining three subunits bind oxygen more easily.

The **Hill coefficient (n)** quantifies cooperativity:

| n value | Meaning | Binding curve shape |
|---------|---------|---------------------|
| **n = 1** | Completely independent binding. Each event has no effect on the others. | Hyperbolic |
| **n ≈ 2.8** | Strong positive cooperativity (real hemoglobin). Early binding events progressively promote later ones. | Sigmoidal |
| **n = 4** | Theoretical all-or-nothing maximum for a 4-subunit protein. The protein would bind zero O₂ until a sharp threshold, then snap fully saturated. | Near step-function |

**Why n ≈ 2.8 and not 4.0?** Each subunit undergoes its own T→R conformational change when oxygen binds, but the four subunits are not perfectly coupled — there is some probability of mixed intermediate states (e.g., two subunits in R-state while two remain in T-state). Real hemoglobin behaves as if approximately 2.8 subunits act cooperatively per cycle, not all four simultaneously.

**The physiological payoff of cooperativity:**

| Location | O₂ partial pressure | Hb saturation |
|---|---|---|
| Lungs (alveoli) | ~100 mmHg | ~97% |
| Resting muscle | ~40 mmHg | ~75% |
| Active muscle | ~20 mmHg | ~30% |

In active muscle, hemoglobin swings from ~97% saturated (lungs) to ~30% saturated (tissues) — delivering ~67% of its oxygen load per cycle. A non-cooperative protein (n = 1) with the same half-saturation point would only deliver ~10–20% per cycle. **Cooperativity makes hemoglobin two to three times more efficient at oxygen delivery.**

### What We Will Do

1. Identify residues at the α–β subunit interface
2. Characterize the chemical properties of those interface residues
3. Estimate interface size using an atom-contact proxy
4. Compare the T-state and R-state structures to measure how the interface changes

**Structures used:**
- **1HHO** — oxy-hemoglobin (R-state), downloaded as the **biological assembly** (full α₂β₂ tetramer, all four chains)
- **2HHB** — deoxy-hemoglobin (T-state, all four chains)

## Learning Objectives

By the end of this notebook, you will be able to:

- Define a protein-protein interface computationally and write a function to find interface residues
- Classify amino acids by chemical property and use that classification to characterize an interface
- Use NumPy broadcasting to efficiently compute pairwise atom distances
- Compare two related structures to quantify a conformational change
- Use pandas `groupby` and `value_counts()` to summarize and compare datasets

**Estimated time:** 60–75 minutes

In [ ]:
# -----------------------------------------------------------------------
# SETUP: Install and import all required libraries
# Run this cell first — it only needs to run once per Colab session
#
# IMPORTANT: After this cell finishes, go to:
#   Runtime → Restart session
# Then run ALL cells again from the top.
# -----------------------------------------------------------------------

!pip install biopython nglview==3.0.8 --quiet

from google.colab import output
output.enable_custom_widget_manager()

import os                          # file and path operations
import requests                    # HTTP requests (used to download the biological assembly)
import numpy as np                 # numeric arrays and fast math
import pandas as pd                # data tables
import matplotlib.pyplot as plt    # charts and graphs
import nglview as nv               # interactive 3D molecular viewer

from Bio.PDB import PDBList, PDBParser    # download and parse PDB files

print('All libraries imported successfully!')
print(f'nglview version: {nv.__version__}')
print()

# -----------------------------------------------------------------------
# Helper function: Euclidean distance between two BioPython atoms
# (same function from Notebook 3, redefined here for standalone use)
# -----------------------------------------------------------------------

def calculate_distance(atom1, atom2):
    '''
    Calculate the Euclidean distance between two atoms in 3D space.

    Parameters:
    -----------
    atom1, atom2 : Bio.PDB.Atom.Atom
        Two atoms, each with a .coord attribute (NumPy array [x, y, z])

    Returns:
    --------
    float : distance in Angstroms (A)
    '''
    diff = atom2.coord - atom1.coord
    return float(np.sqrt(np.sum(diff ** 2)))


print('calculate_distance() function ready.')

In [ ]:
# -----------------------------------------------------------------------
# Download the 1HHO BIOLOGICAL ASSEMBLY and build a 4-chain tetramer
#
# The asymmetric unit of 1HHO contains only the ab heterodimer (chains
# A and B). The R-state tetramer has a molecular 2-fold symmetry axis,
# so RCSB applies a rotation to generate the second ab dimer. That
# rotation is stored in the biological assembly file (.pdb1).
#
# RCSB uses one of two formats in the .pdb1 file:
#
#   Format A: one MODEL record — all four chains labeled A, B, C, D
#   Format B: two MODEL records — each with chains A (alpha) and B (beta)
#             MODEL 0 = original dimer (alpha-1, beta-1)
#             MODEL 1 = rotated copy   (alpha-2, beta-2)
#
# In Format B, BioPython's structure[0] only sees two chains; there is
# no chain C or D. To work around this, we detect Format B and then:
#   1. Copy MODEL 0's chains as A and B (unchanged)
#   2. Copy MODEL 1's chains, renaming A→C and B→D
#   3. Combine all four into a single 4-chain PDB file
#   4. Re-parse and update rstate_path so nglview loads the full tetramer
# -----------------------------------------------------------------------

print('Downloading 1HHO biological assembly (full a2b2 tetramer)...')

biounit_url = 'https://files.rcsb.org/download/1HHO.pdb1'

# The try/except block catches network errors and gives a clear message
try:
    response = requests.get(biounit_url, timeout=30)
except Exception as network_error:
    raise RuntimeError(
        'Could not connect to RCSB PDB to download 1HHO.\n'
        'Check your internet connection and try again.\n'
        f'Original error: {network_error}'
    )

if response.status_code == 200:
    assembly_path = '1hho_assembly.pdb'
    with open(assembly_path, 'w') as f:
        f.write(response.text)
    print(f'Saved raw assembly to: {assembly_path}')
else:
    print(f'Download failed (HTTP {response.status_code}) — check your internet connection.')

parser = PDBParser(QUIET=True)
raw_structure      = parser.get_structure('1HHO_raw', assembly_path)
raw_models_list    = list(raw_structure.get_models())

print(f'MODEL records in assembly file: {len(raw_models_list)}')
print()

# -----------------------------------------------------------------------
# Detect format and, if needed, reconstruct a single 4-chain tetramer
# -----------------------------------------------------------------------

if len(raw_models_list) == 1:
    # Format A — four chains already present with unique IDs
    print('Format A detected: chains A, B, C, D in a single MODEL record.')
    rstate_path = assembly_path

else:
    # Format B — two MODEL records; merge into one 4-chain model
    print('Format B detected: two MODEL records with duplicate chain IDs.')
    print('Reconstructing 4-chain tetramer (renaming MODEL 1 chains A→C, B→D)...')

    from Bio.PDB import PDBIO
    from Bio.PDB.Structure import Structure
    from Bio.PDB.Model import Model as BioModel

    new_struct    = Structure('1HHO_tetramer')
    new_model_obj = BioModel(0)
    new_struct.add(new_model_obj)

    # MODEL 0: keep original chain IDs (A = alpha-1, B = beta-1)
    for chain in raw_structure[0].get_chains():
        new_model_obj.add(chain.copy())

    # MODEL 1: rename chains (A → C = alpha-2, B → D = beta-2)
    rename_map = {'A': 'C', 'B': 'D'}
    for chain in raw_structure[1].get_chains():
        chain_copy    = chain.copy()
        chain_copy.id = rename_map.get(chain.id, chain.id)
        new_model_obj.add(chain_copy)

    rstate_path = '1hho_tetramer_4chain.pdb'
    io = PDBIO()
    io.set_structure(new_struct)
    io.save(rstate_path)
    print(f'4-chain tetramer written to: {rstate_path}')

# -----------------------------------------------------------------------
# Parse the final 4-chain file — used by all cells below
# -----------------------------------------------------------------------

rstate_structure = parser.get_structure('1HHO', rstate_path)
rstate_model     = rstate_structure[0]

print()
print('1HHO tetramer parsed successfully.')
print()
for chain in rstate_model.get_chains():
    aa = [r for r in chain.get_residues() if r.id[0] == ' ']
    print(f'  Chain {chain.id}: {len(aa)} amino acid residues')

print()
print('Chain assignment:')
print('  A = alpha-1   B = beta-1   (original asymmetric unit)')
print('  C = alpha-2   D = beta-2   (symmetry-generated copy, renamed)')
print()

# chain_D is the beta-2 subunit — used for a1b2 interface analysis
chain_D = rstate_model['D']

print('rstate_path is now the 4-chain tetramer file.')
print('All nglview cells will load the complete a2b2 structure with chains A, B, C, D.')


---
## Section 1: Identifying Interface Residues

### What is a Protein-Protein Interface?

When two chains assemble, they bury a patch of their surfaces — atoms that were exposed to water become hidden inside the complex. The residues that contribute to this buried patch are the **interface residues**.

Computationally, finding interface residues is straightforward: for every residue on chain 1 and every residue on chain 2, check whether the two residues are close enough to interact. If they are, both residues are interface residues.

### Our Approach: CA-Based Distance Detection

We use the **alpha carbon (CA)** of each residue as its representative point. If two CA atoms are within **8.0 Å** of each other, we flag both residues as interface residues.

Why 8.0 Å for CA atoms when typical interaction distances are 4–5 Å?

- The CA atom sits in the backbone, not the side chain
- Side chains extend 3–6 Å beyond the backbone
- Two residues whose *side chains* interact will typically have CA–CA distances of 6–10 Å
- An 8.0 Å CA–CA cutoff reliably captures residues that interact at the side-chain level

This is a standard approach in structural bioinformatics when all-atom detection is not required.

### The α1β1 vs. α1β2 Interfaces

| Interface | Chains | Role in Hemoglobin |
|-----------|--------|--------------------|
| α1β1 | A–B | Large **packing** interface — stabilizes the αβ dimer; does **not** change significantly between T and R states |
| α1β2 | A–D | Smaller **sliding** interface — rotates ~15° during T→R transition; this is the **allosteric switch** |

We will analyze the α1β1 interface first (chains A and B), then compare it to α1β2 (chains A and D) in Section 3.

In [ ]:
# -----------------------------------------------------------------------
# Define find_interface_residues()
#
# Strategy:
#   1. Get all standard amino acid residues from each chain
#   2. For each pair of residues (one from each chain), measure the
#      CA-CA distance
#   3. If the distance is <= ca_threshold, mark both residues as
#      interface residues
#
# Using only the CA atom (rather than all atoms) makes this ~100x
# faster with no meaningful loss of accuracy for interface detection.
# -----------------------------------------------------------------------

def find_interface_residues(chain1, chain2, ca_threshold=8.0):
    '''
    Find residues at the protein-protein interface between two chains.

    Uses alpha carbon (CA) distances as a proxy for residue proximity.
    A residue is classified as an interface residue if its CA is within
    ca_threshold Angstroms of any CA atom on the opposite chain.

    Parameters:
    -----------
    chain1 : Bio.PDB.Chain.Chain
        First protein chain
    chain2 : Bio.PDB.Chain.Chain
        Second protein chain
    ca_threshold : float
        Maximum CA-CA distance (A) to classify a residue pair as interface
        residues (default 8.0 A)

    Returns:
    --------
    tuple : (interface_residues_chain1, interface_residues_chain2)
        Each element is a list of Bio.PDB.Residue objects at the interface,
        sorted by residue sequence number.
    '''
    # Get all standard amino acid residues that have a CA atom
    # Skipping residue.id[0] != ' ' removes HETATM records (water, heme, etc.)
    residues1 = [res for res in chain1 if res.id[0] == ' ' and 'CA' in res]
    residues2 = [res for res in chain2 if res.id[0] == ' ' and 'CA' in res]

    # Use sets to collect interface residues — sets automatically avoid duplicates
    interface_set1 = set()
    interface_set2 = set()

    # Check every pair of residues across the two chains
    for res1 in residues1:
        ca1 = res1['CA']
        for res2 in residues2:
            ca2 = res2['CA']
            dist = calculate_distance(ca1, ca2)
            if dist <= ca_threshold:
                interface_set1.add(res1)
                interface_set2.add(res2)

    # Convert to sorted lists (sorted by residue sequence number)
    interface_chain1 = sorted(interface_set1, key=lambda r: r.id[1])
    interface_chain2 = sorted(interface_set2, key=lambda r: r.id[1])

    return interface_chain1, interface_chain2


print('find_interface_residues() function defined.')
print('Will use CA-CA distance threshold of 8.0 A by default.')

In [ ]:
# -----------------------------------------------------------------------
# Apply find_interface_residues() to the a1b1 interface (chains A and B)
# -----------------------------------------------------------------------

print('Analyzing the a1b1 interface (Chain A [alpha-1] <-> Chain B [beta-1])...')
print('CA-CA threshold: 8.0 A')
print()

chain_A = rstate_model['A']   # alpha-1 subunit
chain_B = rstate_model['B']   # beta-1 subunit

interface_A, interface_B = find_interface_residues(chain_A, chain_B, ca_threshold=8.0)

print(f'Interface residues on Chain A (alpha-1): {len(interface_A)}')
print(f'Interface residues on Chain B (beta-1) : {len(interface_B)}')
print(f'Total interface residues                : {len(interface_A) + len(interface_B)}')
print()

# Build a DataFrame — one row per interface residue
interface_records = []

for res in interface_A:
    interface_records.append({
        'Chain':   'A (alpha-1)',
        'ResNum':  res.id[1],
        'ResName': res.get_resname()
    })

for res in interface_B:
    interface_records.append({
        'Chain':   'B (beta-1)',
        'ResNum':  res.id[1],
        'ResName': res.get_resname()
    })

interface_df = pd.DataFrame(interface_records)
interface_df = interface_df.sort_values(['Chain', 'ResNum']).reset_index(drop=True)

print('a1b1 interface residues (first 20 rows):')
print(interface_df.head(20).to_string(index=False))
print()
print(f'... and {max(0, len(interface_df) - 20)} more rows.')
print()

# Quick summary: most common residue types at the interface
print('Most frequent residue types at the a1b1 interface:')
print(interface_df['ResName'].value_counts().head(10).to_string())

In [ ]:
# -----------------------------------------------------------------------
# Visualize the a1b1 interface residues in 3D (nglview)
#
# rstate_path now points to the 4-chain tetramer file (chains A, B, C, D),
# so we can use distinct colors for all four subunits.
#
# Visualization layers:
#   1. Gray cartoon backdrop — chains C and D (the second dimer)
#   2. Blue cartoon — chain A (alpha-1)
#   3. Red  cartoon — chain B (beta-1)
#   4. Blue ball-and-stick — alpha-1 interface residues (chain A)
#   5. Red  ball-and-stick — beta-1  interface residues (chain B)
#
# The interface residues form a continuous patch where chains A and B
# contact each other. Rotate the structure to see it clearly.
# -----------------------------------------------------------------------

from IPython.display import display


def build_ngl_selection(residue_list, chain_id):
    '''
    Build an NGL viewer selection string for a list of residues.

    NGL selection format: "42:A or 44:A or 88:A"

    Parameters:
    -----------
    residue_list : list of Bio.PDB.Residue
        Residues to include in the selection
    chain_id : str
        Single-letter chain identifier (e.g. 'A', 'B')

    Returns:
    --------
    str : NGL selection string
    '''
    resnums = [str(r.id[1]) for r in residue_list]
    return ' or '.join([f'{n}:{chain_id}' for n in resnums])


iface_sel_A = build_ngl_selection(interface_A, 'A')
iface_sel_B = build_ngl_selection(interface_B, 'B')

print('Visualizing the a1b1 interface residues in 3D...')
print(f'  Chain A interface residues highlighted: {len(interface_A)}')
print(f'  Chain B interface residues highlighted: {len(interface_B)}')
print()
print('Color key:')
print('  Blue  = Chain A (alpha-1): cartoon + ball-and-stick interface residues')
print('  Red   = Chain B (beta-1) : cartoon + ball-and-stick interface residues')
print('  Gray  = Chains C and D   : spatial context (the second ab dimer)')

view = nv.show_structure_file(rstate_path, representations=[])
view.background = 'white'

# Layer 1: gray cartoon backdrop for the second dimer (spatial context)
view.add_representation('cartoon', selection=':C or :D', color='#CCCCCC')

# Layer 2: colored cartoons for the a1b1 chains
view.add_representation('cartoon', selection=':A', color='#2166AC')
view.add_representation('cartoon', selection=':B', color='#D7191C')

# Layer 3: ball-and-stick highlights for the specific interface residues
view.add_representation('ball+stick', selection=iface_sel_A, color='#2166AC')
view.add_representation('ball+stick', selection=iface_sel_B, color='#D7191C')

display(view)

### 💭 Think About It

1. **Count the interface residues.** The alpha and beta subunits of hemoglobin each have ~141 residues. What fraction of each chain is at the α1β1 interface? Is this more or less than you expected for a stable protein-protein interface?

2. **Look at the most common residue types.** Do you see more hydrophobic residues (ALA, VAL, LEU, ILE, PHE) or more charged residues (ARG, LYS, ASP, GLU) in the interface list? Based on what you know from Notebook 3, what does this suggest about the primary force holding the α1β1 interface together?

3. **The α1β1 interface is described as the "packing" interface** — large and resistant to conformational change. What structural property would make an interface stable and rigid? Think about which types of interactions are hardest to break reversibly.

4. **Hemoglobin has two α1β1 interfaces** (A–B and C–D) **and two α1β2 interfaces** (A–D and C–B). Why might a protein with one large stable interface AND one smaller flexible regulatory interface be functionally superior to a protein where all interfaces are equally strong?

---
## Section 2: Characterizing the Interface

### Why Does Interface Composition Matter?

The types of amino acids at an interface determine what forces hold it together and how it might respond to cellular signals:

- **Predominantly hydrophobic interfaces** are driven by the hydrophobic effect — stable and rigid, because burying nonpolar surface is energetically favorable
- **Interfaces rich in charged residues** are held together by ionic interactions and are often **regulatable**: pH changes, phosphorylation, or ion concentration changes can alter them
- **Interfaces with mixed composition** appear at allosteric sites — strong enough to maintain the complex but flexible enough to switch conformation

### Amino Acid Classification

We will assign each of the 20 standard amino acids to one of three categories:

| Category | Amino acids |
|----------|-------------|
| **Hydrophobic** | ALA, VAL, ILE, LEU, MET, PHE, TRP, PRO |
| **Polar (uncharged)** | SER, THR, CYS, TYR, ASN, GLN, GLY, **HIS** |
| **Charged** | ARG, LYS *(positive)*, ASP, GLU *(negative)* |

> **A note on Histidine (HIS):** His has a side chain pKa of ~6.0, which means it is mostly uncharged (~91% neutral) at physiological pH 7.4. For that reason it is placed in the Polar category here — the standard convention in many structural bioinformatics tools. However, His *can* carry a positive charge at slightly acidic pH (lysosomes, ischemic tissue, enzyme active sites), and some textbooks group it with Arg and Lys as a basic residue. This ambiguity is why His appears so often in enzyme active sites: it can act as both a proton donor and acceptor near physiological pH, making it uniquely suited for catalysis.

We will compare the composition of the **α1β1 interface** residues to the composition of **all residues in the hemoglobin tetramer** to see if the interface is enriched or depleted in any category.

In [ ]:
# -----------------------------------------------------------------------
# Classify amino acids by chemical property
# -----------------------------------------------------------------------

HYDROPHOBIC_RESIDUES = {'ALA', 'VAL', 'ILE', 'LEU', 'MET', 'PHE', 'TRP', 'PRO'}
POLAR_RESIDUES       = {'SER', 'THR', 'CYS', 'TYR', 'ASN', 'GLN', 'GLY', 'HIS'}
CHARGED_RESIDUES     = {'ARG', 'LYS', 'ASP', 'GLU'}


def classify_residue(resname):
    '''
    Classify a three-letter residue code by chemical property.

    Parameters:
    -----------
    resname : str
        Three-letter residue code (e.g. 'ALA', 'GLU')

    Returns:
    --------
    str : 'Hydrophobic', 'Polar', 'Charged', or 'Other'
    '''
    if resname in HYDROPHOBIC_RESIDUES:
        return 'Hydrophobic'
    elif resname in POLAR_RESIDUES:
        return 'Polar'
    elif resname in CHARGED_RESIDUES:
        return 'Charged'
    else:
        return 'Other'   # non-standard residue or unclassified


# --- Classify the interface residues ---
# We add a 'Property' column directly to the existing interface_df
interface_df['Property'] = interface_df['ResName'].apply(classify_residue)

print('a1b1 interface composition by chemical property:')
print('=' * 50)
interface_total  = len(interface_df)
interface_counts = interface_df['Property'].value_counts()
for prop, count in interface_counts.items():
    pct = 100 * count / interface_total
    print(f'  {prop:<15}: {count:3d} residues  ({pct:.1f}%)')
print()

# --- Classify ALL residues in the hemoglobin tetramer ---
all_residue_records = []
for chain in rstate_model.get_chains():
    for residue in chain:
        if residue.id[0] != ' ':
            continue   # skip HETATM records (water, heme, etc.)
        all_residue_records.append({
            'Chain':    chain.id,
            'ResNum':   residue.id[1],
            'ResName':  residue.get_resname(),
            'Property': classify_residue(residue.get_resname())
        })

all_res_df = pd.DataFrame(all_residue_records)

print('Overall hemoglobin tetramer composition by chemical property:')
print('=' * 50)
all_total  = len(all_res_df)
all_counts = all_res_df['Property'].value_counts()
for prop, count in all_counts.items():
    pct = 100 * count / all_total
    print(f'  {prop:<15}: {count:3d} residues  ({pct:.1f}%)')

In [ ]:
# -----------------------------------------------------------------------
# Stacked bar chart: interface composition vs. overall tetramer composition
#
# Each bar is divided into three segments (Hydrophobic / Polar / Charged).
# The percentage labels are placed in the center of each segment.
# The legend is placed outside the plot area so it does not overlap the bars.
# -----------------------------------------------------------------------

categories  = ['Hydrophobic', 'Polar', 'Charged']
bar_colors  = ['#ED7D31', '#4472C4', '#FF0000']

# Calculate percentages for the interface and for the full tetramer
iface_pct   = [100 * interface_df['Property'].value_counts().get(c, 0) / len(interface_df)   for c in categories]
overall_pct = [100 * all_res_df['Property'].value_counts().get(c, 0)   / len(all_res_df)     for c in categories]

fig, ax = plt.subplots(figsize=(8, 6))

bottom_iface   = 0
bottom_overall = 0

for cat, color, ip, op in zip(categories, bar_colors, iface_pct, overall_pct):
    # Draw the two bars side by side at x=0 and x=1
    ax.bar(0, ip, bottom=bottom_iface,   color=color, edgecolor='black',
           linewidth=0.5, width=0.5, label=cat)
    ax.bar(1, op, bottom=bottom_overall, color=color, edgecolor='black',
           linewidth=0.5, width=0.5)

    # Add a percentage label inside each segment (only if wide enough to read)
    if ip > 6:
        ax.text(0, bottom_iface   + ip / 2, f'{ip:.0f}%',
                ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    if op > 6:
        ax.text(1, bottom_overall + op / 2, f'{op:.0f}%',
                ha='center', va='center', fontsize=11, fontweight='bold', color='white')

    bottom_iface   += ip
    bottom_overall += op

ax.set_xticks([0, 1])
ax.set_xticklabels(['a1b1 Interface\nresidues', 'All tetramer\nresidues'], fontsize=12)
ax.set_ylabel('Percentage of residues (%)', fontsize=12)
ax.set_title('Amino Acid Composition: a1b1 Interface vs. Full Tetramer\n(1HHO, oxy-hemoglobin)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)

# Place legend to the right of the plot so it does not overlap the bars
ax.legend(title='Property', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11)

ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 💭 Think About It

1. **Is the α1β1 interface enriched in hydrophobic residues** compared to the overall protein? Interfaces that are primarily hydrophobic tend to be stable and rigid — burying nonpolar surface is energetically favorable. Does the composition you found match the description of the α1β1 interface as a "stable packing interface"?

2. **Charged residues on the protein surface** help keep the protein soluble by interacting with water. If the interface is depleted in charged residues, what does this tell you about which part of the protein surface is buried at the interface?

3. **Compare to an antibody-antigen interface:** antibody-antigen recognition interfaces tend to be enriched in charged and polar residues (specific H-bonds and ionic interactions enable recognition of a particular antigen). The hemoglobin α1β1 interface is a permanent assembly. Why might permanent interfaces trend toward hydrophobic character while transient recognition interfaces trend polar/charged?

4. **Glycine (GLY) has no side chain at all** — it was placed in the Polar category here as a default. How should GLY really be classified? What types of interactions can a Gly at an interface actually form? How would reclassifying it affect the bar chart?

---
## Section 3: Calculating Contact Area (Simplified)

### What is Buried Surface Area?

When two chains assemble, they remove a patch of surface from contact with water. The **buried surface area (BSA)** quantifies how much surface is hidden — it is the single most important predictor of interface stability. Larger BSA = stronger, more stable interface.

The formal calculation rolls a "probe" sphere (radius 1.4 Å, mimicking a water molecule) over the protein surface to compute the **solvent-accessible surface area (SASA)** of each chain alone and in the complex:

> BSA = SASA(chain A alone) + SASA(chain B alone) − SASA(complex)

This is accurate but computationally expensive. We will use a simpler **atom-contact proxy** instead.

### Our Proxy: Atom-Pair Contact Counting

We count the number of atom pairs — one atom from each chain — that are within **4.5 Å** of each other. This is proportional to BSA: more buried atoms means more close contacts. It is not a true surface area, but it correctly ranks which interface is larger.

### NumPy Broadcasting

For a chain with ~1,100 atoms, checking every atom against every atom across two chains means ~1.2 million distance calculations. A Python loop would be slow; we use **NumPy broadcasting** instead:

1. Collect all atom coordinates from chain 1 into an array of shape **(N₁ × 3)**
2. Collect all atom coordinates from chain 2 into an array of shape **(N₂ × 3)**
3. Subtract every row of one array from every row of the other — broadcasting produces an **(N₁ × N₂ × 3)** difference matrix in one step
4. Compute the Euclidean length of each difference vector → an **(N₁ × N₂)** distance matrix
5. Count entries ≤ threshold

This takes a fraction of a second rather than minutes.

### What We Will Compare

| Interface | Chains | Expected size | Role |
|-----------|--------|---------------|------|
| α1β1 | A–B | Larger | Stable packing interface |
| α1β2 | A–D | Smaller | Allosteric switch interface |

In [ ]:
# -----------------------------------------------------------------------
# Define count_interface_contacts()
#
# Instead of computing true buried surface area (which requires a rolling
# probe), we count atom pairs within a distance threshold. This serves as
# a fast, interpretable proxy for interface size.
# -----------------------------------------------------------------------

def count_interface_contacts(chain1, chain2, threshold=4.5):
    '''
    Count atom-atom contacts across the interface between two chains.

    Uses NumPy broadcasting to compute all pairwise distances in one
    vectorized operation — much faster than a Python nested loop.

    Parameters:
    -----------
    chain1 : Bio.PDB.Chain.Chain
        First protein chain
    chain2 : Bio.PDB.Chain.Chain
        Second protein chain
    threshold : float
        Maximum atom-atom distance (A) to count as a contact (default 4.5 A)

    Returns:
    --------
    int : total number of atom pairs within the threshold distance
    '''
    # Step 1: Extract all atom coordinates from standard amino acids in each chain
    # List comprehension: for each residue (if it is a standard AA), for each atom
    coords1 = np.array([
        atom.coord
        for residue in chain1
        if residue.id[0] == ' '      # standard amino acid only
        for atom in residue
    ])

    coords2 = np.array([
        atom.coord
        for residue in chain2
        if residue.id[0] == ' '
        for atom in residue
    ])

    print(f'  Chain {chain1.id}: {len(coords1):,} atoms')
    print(f'  Chain {chain2.id}: {len(coords2):,} atoms')
    print(f'  Checking {len(coords1) * len(coords2):,} atom pairs with NumPy broadcasting...')

    # Step 2: Broadcasting subtraction
    # coords1[:, np.newaxis, :] has shape (N1, 1, 3)
    # coords2[np.newaxis, :, :] has shape (1, N2, 3)
    # Subtraction broadcasts to shape              (N1, N2, 3)
    diff_matrix = coords1[:, np.newaxis, :] - coords2[np.newaxis, :, :]

    # Step 3: Euclidean length of each difference vector -> shape (N1, N2)
    dist_matrix = np.sqrt(np.sum(diff_matrix ** 2, axis=2))

    # Step 4: Count pairs within the threshold
    contact_count = int(np.sum(dist_matrix <= threshold))

    return contact_count


print('count_interface_contacts() function defined.')

In [ ]:
# -----------------------------------------------------------------------
# Compare the a1b1 and a1b2 interfaces by atom contact count
#
# Both interfaces are measured in the R-state (1HHO biological assembly).
# chain_D (beta-2) was set up in the download cell to work regardless
# of whether the assembly uses Format A (single model, chain D) or
# Format B (two models, chain B from MODEL 1).
# -----------------------------------------------------------------------

print('Counting atom contacts at each interface (threshold: 4.5 A)...')
print()

# chain_A and chain_B are defined below from rstate_model.
# chain_D (beta-2) was already set in the download cell.

# --- a1b1 interface: chains A (alpha-1) and B (beta-1) ---
print('a1b1 interface [alpha-1 x beta-1]:')
contacts_a1b1 = count_interface_contacts(chain_A, chain_B, threshold=4.5)
print(f'  Result: {contacts_a1b1:,} atom contacts')
print()

# --- a1b2 interface: chains A (alpha-1) and chain_D (beta-2) ---
print('a1b2 interface [alpha-1 x beta-2]:')
contacts_a1b2 = count_interface_contacts(chain_A, chain_D, threshold=4.5)
print(f'  Result: {contacts_a1b2:,} atom contacts')
print()

# --- Summarize in a comparison DataFrame ---
comparison_df = pd.DataFrame({
    'Interface':     ['a1b1', 'a1b2'],
    'Chains':        ['A-B',  'A-D'],
    'Role':          ['Stable packing interface', 'Allosteric switch interface'],
    'Atom_contacts': [contacts_a1b1, contacts_a1b2]
})

print('Interface comparison summary:')
print('=' * 65)
print(comparison_df.to_string(index=False))
print()

ratio = contacts_a1b1 / contacts_a1b2
print(f'The a1b1 interface has {ratio:.1f}x more atom contacts than a1b2.')
print()
print('This is consistent with the known biology:')
print('  a1b1 = large, stable interface (~30 contact residue pairs in the literature)')
print('  a1b2 = smaller, flexible interface that rotates ~15 degrees during allostery')

### 💭 Think About It

1. **The α1β1 interface has more contacts than α1β2** — typically about twice as many. The α1β1 interface holds the αβ dimer together as a rigid unit; α1β2 is the allosteric switch. Why would it be advantageous for the allosteric interface (α1β2) to be *smaller*? What would happen to cooperativity if α1β2 were as large and rigid as α1β1?

2. **NumPy broadcasting** computed ~1 million pairwise distances in a fraction of a second. If you were analyzing a large protein complex with 5,000 atoms per chain, how many pairs would you need to compute? Estimate how long a Python nested loop would take at approximately 100,000 pairs per second.

3. **Our atom-count proxy is not the same as buried surface area.** Two interfaces could have the same contact count but very different buried surface areas — for example, if one has a large, flat surface and the other has a small, closely packed pocket. What additional geometric information would you need to distinguish these cases?

4. **Hemoglobin has two α1β2 interfaces** (A–D and C–B) that both change during allostery. If you wanted to measure the total conformational rearrangement of the entire tetramer, would you just add up all interface contact counts? What would you be double-counting, if anything?

---
## Section 4: T-state vs. R-state — Visualizing the Allosteric Switch

### Two Conformations of the Same Tetramer

Hemoglobin exists in dynamic equilibrium between two quaternary conformations:

| State | Name | Oxygen affinity | When dominant |
|-------|------|----------------|---------------|
| **T-state** | Tense | Low | Deoxygenated (tissues, low O₂) |
| **R-state** | Relaxed | High | Oxygenated (lungs, high O₂) |

The T→R transition involves a **~15° rotation** of one αβ dimer relative to the other. This is a small rotation in absolute displacement (2–4 Å for many atoms), but it is mechanically coupled to the heme pocket: in the T-state the iron atom sits slightly out of the heme plane (under strain), and oxygen binding pulls it into the plane, triggering the rotation.

### The Key Allosteric Contact: Tyr42α — Asp99β

At the α1β2 interface, one residue pair defines the T-state:

- **T-state (2HHB, deoxy):** Tyr42 on the alpha chain forms a **hydrogen bond** to Asp99 on the *opposite* beta chain. This bond holds the tetramer in the low-affinity conformation.
- **R-state (1HHO, oxy):** this hydrogen bond is **broken** — the two residues have moved apart, releasing the constraint and allowing all four subunits to bind oxygen with higher affinity.

Measuring the CA–CA distance between Tyr42 (chain A) and Asp99 (chain D) in both structures gives us a direct computational readout of this conformational switch.

> **Why chain A and chain D?** The α1β2 interface is between alpha-1 (chain A) and beta-2 (chain D) — two subunits from *different* αβ dimers. This cross-dimer contact is what transmits the allosteric signal across the whole tetramer.

### What We Will Compute

We will measure CA–CA distances for three key residue pairs at the α1β2 interface in **both states**:

| Residue pair | Expected change |
|---|---|
| **Tyr42(α) — Asp99(β)** | Key allosteric H-bond — distance *increases* in R-state as bond breaks |
| **Thr41(α) — His97(β)** | FG-corner contact — shifts as the interface rotates |
| **Pro44(α) — Phe41(β)** | Hydrophobic contact in the switch region — also shifts |

Because we downloaded the **1HHO biological assembly**, `rstate_model` contains all four chains including chain D. Combined with the 2HHB T-state structure (which also stores all four chains), we can measure the same residue pairs in both states and directly quantify how the α1β2 interface changes during allostery.

In [ ]:
# -----------------------------------------------------------------------
# Download and parse deoxy-hemoglobin (2HHB) — the T-state structure
#
# 2HHB is the classic Perutz deoxy-hemoglobin structure (2.5 A, 1984).
# Unlike 1HHO, the T-state tetramer lacks a clean 2-fold crystallographic
# symmetry axis, so all four chains are stored explicitly:
#   A = alpha-1, B = beta-1, C = alpha-2, D = beta-2
#
# We use PDBList (from BioPython) to download the standard PDB file.
# PDBList is a separate class from PDBParser:
#   PDBList  — handles downloading files from RCSB
#   PDBParser — handles parsing/reading PDB files into Python objects
# -----------------------------------------------------------------------

from Bio.PDB import PDBList

pdb_downloader = PDBList()   # create a downloader object

print('Downloading T-state hemoglobin (2HHB, deoxy)...')

# The try/except block catches network errors and gives a clear message
try:
    tstate_path = pdb_downloader.retrieve_pdb_file('2HHB', file_format='pdb', pdir='.')
except Exception as download_error:
    raise RuntimeError(
        'Could not download 2HHB from RCSB PDB.\n'
        'Check your internet connection and try again.\n'
        f'Original error: {download_error}'
    )

print(f'Downloaded to: {tstate_path}')

# BioPython sometimes saves the file in a subdirectory; search if not found
if not os.path.exists(tstate_path):
    print('File not at expected path, searching...')
    for root, dirs, files in os.walk('.'):
        for fname in files:
            if '2hhb' in fname.lower():
                tstate_path = os.path.join(root, fname)
                print(f'Found at: {tstate_path}')
                break

tstate_structure = parser.get_structure('2HHB', tstate_path)
tstate_model     = tstate_structure[0]

print()
print('2HHB (deoxy-hemoglobin, T-state) parsed successfully.')
print()

for chain in tstate_model.get_chains():
    aa_residues = [r for r in chain.get_residues() if r.id[0] == ' ']
    print(f'  Chain {chain.id}: {len(aa_residues)} amino acid residues')

print()
print('All four chains are stored explicitly (T-state lacks 2-fold symmetry):')
print('  A = alpha-1, B = beta-1, C = alpha-2, D = beta-2')


In [ ]:
# -----------------------------------------------------------------------
# Visualize T-state (2HHB) and R-state (1HHO) side by side
#
# Both structures now have four explicitly named chains (A, B, C, D):
#   2HHB stores all four chains in the PDB file natively (T-state lacks
#   the symmetry shortcut, so RCSB writes them all out explicitly).
#   1HHO was reconstructed from the biological assembly in cell-03 to
#   give a single 4-chain tetramer with the same A/B/C/D naming.
#
# Consistent color scheme across both viewers:
#   Chain A (alpha-1): blue   (#2166AC)
#   Chain B (beta-1) : red    (#D7191C)
#   Chain C (alpha-2): orange (#F4A811)
#   Chain D (beta-2) : green  (#1A9641)
#
# Try to spot the ~15-degree rotation of the orange-green dimer relative
# to the blue-red dimer between the T-state and R-state viewers.
# Orient the tetramer end-on (looking down the 2-fold axis) for the
# clearest view of the allosteric rotation.
# -----------------------------------------------------------------------

from IPython.display import display

print('T-state: deoxy-hemoglobin (2HHB, low O2 affinity)')
print('Colors:  Blue=A (alpha-1)  Red=B (beta-1)  Orange=C (alpha-2)  Green=D (beta-2)')
print()

view_t = nv.show_structure_file(tstate_path, representations=[])
view_t.background = 'white'
view_t.add_representation('cartoon', selection=':A', color='#2166AC')
view_t.add_representation('cartoon', selection=':B', color='#D7191C')
view_t.add_representation('cartoon', selection=':C', color='#F4A811')
view_t.add_representation('cartoon', selection=':D', color='#1A9641')
display(view_t)

print()
print('R-state: oxy-hemoglobin (1HHO, high O2 affinity)')
print('Same color scheme. The tetramer was reconstructed from the biological assembly')
print('so chains C and D are the symmetry-generated alpha-2 and beta-2 subunits.')
print()

view_r = nv.show_structure_file(rstate_path, representations=[])
view_r.background = 'white'
view_r.add_representation('cartoon', selection=':A', color='#2166AC')
view_r.add_representation('cartoon', selection=':B', color='#D7191C')
view_r.add_representation('cartoon', selection=':C', color='#F4A811')
view_r.add_representation('cartoon', selection=':D', color='#1A9641')
display(view_r)

In [ ]:
# -----------------------------------------------------------------------
# Measure a1b2 interface distances: T-state (2HHB) vs R-state (1HHO)
#
# We measure CA-CA distances for three documented contact pairs at the
# a1b2 interface (alpha-1 vs beta-2) in both states.
#
# T-state: tstate_model['A'] (alpha-1) and tstate_model['D'] (beta-2)
#          from 2HHB — which stores all four chains explicitly.
#
# R-state: rstate_model['A'] (alpha-1) and chain_D (beta-2)
#          from 1HHO biological assembly — chain_D was set in the
#          download cell to handle both Format A and Format B files.
# -----------------------------------------------------------------------

contact_pairs = [
    (42,  99, 'Tyr42(a) - Asp99(b)  [key allosteric H-bond]'),
    (41,  97, 'Thr41(a) - His97(b)  [FG-corner contact]'),
    (44,  41, 'Pro44(a) - Phe41(b)  [hydrophobic contact]'),
]

print('a1b2 interface CA distances: T-state vs R-state direct comparison')
print('  T-state: 2HHB (deoxy-hemoglobin, low O2 affinity)')
print('  R-state: 1HHO biological assembly (oxy-hemoglobin, high O2 affinity)')
print('  alpha-1 (chain A) x beta-2 (chain_D) in both structures')
print('=' * 68)
print()

comparison_records = []

for alpha_num, beta_num, label in contact_pairs:
    dist_t = None
    dist_r = None

    # --- T-state distance (from 2HHB, which has chain D explicitly) ---
    try:
        ca_a_t = tstate_model['A'][(' ', alpha_num, ' ')]['CA']
        ca_d_t = tstate_model['D'][(' ', beta_num,  ' ')]['CA']
        dist_t = round(calculate_distance(ca_a_t, ca_d_t), 2)
    except KeyError as e:
        print(f'  Warning (T-state): residue or atom not found — {e}')

    # --- R-state distance (from 1HHO biological assembly) ---
    # chain_D points to beta-2, correctly resolved in the download cell
    # regardless of whether the file uses Format A or Format B.
    try:
        ca_a_r = rstate_model['A'][(' ', alpha_num, ' ')]['CA']
        ca_d_r = chain_D[(' ', beta_num, ' ')]['CA']
        dist_r = round(calculate_distance(ca_a_r, ca_d_r), 2)
    except KeyError as e:
        print(f'  Warning (R-state): residue or atom not found — {e}')

    if dist_t is not None and dist_r is not None:
        delta = round(dist_r - dist_t, 2)
        comparison_records.append({
            'Residue pair':      label,
            'T-state 2HHB (A)':  dist_t,
            'R-state 1HHO (A)':  dist_r,
            'Delta (A)':         delta
        })

if comparison_records:
    comp_df = pd.DataFrame(comparison_records)
    print(comp_df.to_string(index=False))
    print()
    print('Delta = R-state distance minus T-state distance')
    print('  Positive delta (+): residues MOVE APART during T -> R transition')
    print('  Negative delta (-): residues MOVE CLOSER during T -> R transition')
    print()
    print('Tyr42(a)-Asp99(b): the positive delta reflects the H-bond breaking.')
    print('  In T-state these CA atoms are close enough for a side-chain H-bond.')
    print('  In R-state the ~15-degree rotation pulls them apart, releasing the')
    print('  low-affinity constraint and increasing O2 affinity across the tetramer.')
    print()
    print('Note: CA-CA distances do not directly confirm an H-bond is present')
    print('(that requires side-chain geometry), but a large positive delta for')
    print('Tyr42-Asp99 is the standard computational signature of the T->R switch.')
else:
    print('Warning: no comparison records generated.')
    print('Check that tstate_model (2HHB) and rstate_model (1HHO) are both loaded.')

### 💭 Think About It

1. **The Tyr42α–Asp99β distance increases in the R-state** — the hydrogen bond that constrained the T-state is broken when oxygen binds. But this residue pair is on the *opposite side of the tetramer* from where oxygen binds. How does oxygen binding at the heme (deep inside each subunit) cause a conformational change ~20 Å away at the α1β2 interface? What is the molecular "lever" that transmits the signal?

2. **Look at all three residue pairs in the comparison.** Do all three move in the same direction (all positive Δ, all get larger)? Or is the change more complex? What would it mean biologically if some contacts break while new ones form during the T→R transition?

3. **The Hill coefficient for hemoglobin is ~2.8**, not 4.0 (which would be perfect cooperativity). A coefficient of 1.0 means no cooperativity; 4.0 means the first three oxygen molecules *must* bind before the last site can fill. What does n ≈ 2.8 tell you about whether each oxygen-binding event produces exactly the same conformational change?

4. **The T↔R transition is fully reversible.** When hemoglobin releases oxygen in the tissues, it reverts from R to T. What provides the thermodynamic driving force for the reverse transition? (Hint: consider Le Chatelier's principle — what happens to the T↔R equilibrium when oxygen concentration drops?)

---
## Summary

In this notebook, you:

- **Identified interface residues** at the hemoglobin α1β1 complex using `find_interface_residues()`, which uses CA–CA distance (≤ 8.0 Å) as a fast and reliable residue proximity criterion
- **Characterized the interface** by amino acid chemical property (hydrophobic, polar, charged) and compared interface composition to the overall tetramer using a stacked bar chart — finding the α1β1 interface is enriched in hydrophobic residues relative to the full tetramer
- **Estimated interface size** using a simplified atom-contact proxy implemented with NumPy broadcasting — confirming that the α1β1 (packing) interface has substantially more contacts than the α1β2 (allosteric switch) interface
- **Compared T-state and R-state structures** by downloading both 2HHB and 1HHO, measuring key CA distances at the α1β2 interface, and building a before/after comparison DataFrame that shows the allosteric Tyr42α–Asp99β hydrogen bond breaking in the R-state
- **Applied pandas** `value_counts()` and `groupby`-style operations to summarize residue compositions and compare datasets

### Looking Ahead

In **Notebook 5**, we reach the final integration — enzyme active sites. We will examine a catalytic enzyme, identify conserved catalytic residues across species using sequence alignment, and draw on tools from all four prior notebooks (sequence retrieval, secondary structure, 3D distances, and interface analysis) to build a complete structure-function picture.

---
## Try It Yourself: Interface Analysis of a Homo-dimer

A **homo-dimer** is a protein complex made of two identical chains. Unlike hemoglobin's heterologous α–β interfaces, the two chains at a homo-dimer interface have exactly the same sequence — and the interaction is perfectly symmetric.

Homo-dimers are extremely common: transcription factors, metabolic enzymes, signaling proteins, and many others function as homo-dimers. The same surface patch on both copies buries itself at the dimer interface.

### Suggested structure: Triosephosphate Isomerase (TIM)

**PDB ID: `1TIM`** — triosephosphate isomerase from chicken muscle

- Two identical chains (A and B), each ~248 residues
- A beautifully symmetric dimer interface
- One of the most studied enzymes: catalyzes the isomerization step in glycolysis
- Its fold — the **TIM barrel** (α/β barrel) — is the most common enzyme fold in nature

### Your Tasks

1. **Download and parse 1TIM.** How many residues are in each chain? Are chains A and B the same length?

2. **Run `find_interface_residues(chain_A, chain_B, ca_threshold=8.0)`.** Because TIM is a symmetric homo-dimer, the number of interface residues on chain A and chain B should be equal (or nearly equal). Is that what you observe?

3. **Classify interface residues** by chemical property (hydrophobic / polar / charged) and compare the interface composition to the overall dimer composition using a stacked bar chart. Is the TIM dimer interface more hydrophobic or more polar than hemoglobin's α1β1 interface?

4. **Count atom contacts** using `count_interface_contacts()`. Is the TIM dimer interface larger or smaller than the hemoglobin α1β1 interface?

5. **Visualize the TIM dimer in nglview** using `colorScheme='chainindex'`. The two chains should appear as two distinct colors. Can you identify where the interface is? Try adding a `surface` representation at low opacity to see the molecular shape.

> **Hint:** All functions defined in this notebook (`find_interface_residues`, `count_interface_contacts`, `classify_residue`, `calculate_distance`) work on any BioPython structure object — you do not need to modify them. Just call them with the 1TIM structure.

In [ ]:
# -----------------------------------------------------------------------
# Starter code for the TIM homo-dimer interface analysis
# Uncomment and fill in each step below.
# -----------------------------------------------------------------------

# Step 1: Download and parse 1TIM
# tim_path = pdbl.retrieve_pdb_file('1TIM', file_format='pdb', pdir='.')
# if not os.path.exists(tim_path):
#     for root, dirs, files in os.walk('.'):
#         for fname in files:
#             if '1tim' in fname.lower():
#                 tim_path = os.path.join(root, fname)
#                 break
# tim_structure = parser.get_structure('1TIM', tim_path)
# tim_model     = tim_structure[0]
# for chain in tim_model.get_chains():
#     aa_residues = [r for r in chain.get_residues() if r.id[0] == ' ']
#     print(f'Chain {chain.id}: {len(aa_residues)} residues')

# Step 2: Find interface residues
# tim_chain_A = tim_model['A']
# tim_chain_B = tim_model['B']
# tim_iface_A, tim_iface_B = find_interface_residues(tim_chain_A, tim_chain_B, ca_threshold=8.0)
# print(f'Interface residues on Chain A: {len(tim_iface_A)}')
# print(f'Interface residues on Chain B: {len(tim_iface_B)}')

# Step 3: Classify by property and plot
# tim_iface_records = []
# for res in tim_iface_A + tim_iface_B:
#     tim_iface_records.append({
#         'ResName':  res.get_resname(),
#         'Property': classify_residue(res.get_resname())
#     })
# tim_iface_df = pd.DataFrame(tim_iface_records)
# print(tim_iface_df['Property'].value_counts())
# (add your bar chart code here)

# Step 4: Count atom contacts
# tim_contacts = count_interface_contacts(tim_chain_A, tim_chain_B, threshold=4.5)
# print(f'TIM dimer atom contacts       : {tim_contacts:,}')
# print(f'Hemoglobin a1b1 atom contacts : {contacts_a1b1:,}')

# Step 5: Visualize in nglview
# tim_view = nv.show_structure_file(
#     tim_path,
#     representations=[{'type': 'cartoon', 'params': {'colorScheme': 'chainindex'}}]
# )
# tim_view.background = 'white'
# tim_view

# Your code here:
